# ODI to Databricks Migration

**Source File:** `WC_BADGE_D.sql`

**Conversion Timestamp:** 2024-07-30T12:00:00Z

**Description:** This notebook processes badge data, loading it incrementally into the `WC_BADGE_DETAILS_D` target table.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "", "ETL Job Type")
dbutils.widgets.text("DATASOURCE_NUM_ID", "1", "Datasource Number ID")
dbutils.widgets.text("ETL_PROC_WID", "12345", "ETL Process Widget ID")
dbutils.widgets.text("ODI_SESS_NO", "0", "ODI Session Number")

## ETL Parameters

In [ ]:
-- MAGIC %sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_parameters AS
SELECT
  '${ETL_JOB_TYPE}' AS etl_job_type,
  CAST('${DATASOURCE_NUM_ID}' AS BIGINT) AS datasource_num_id,
  CAST('${ETL_PROC_WID}' AS BIGINT) AS etl_proc_wid,
  CAST('${ODI_SESS_NO}' AS BIGINT) AS odi_sess_no;

In [ ]:
display(spark.sql("SELECT *
FROM v_etl_parameters"))

## Staging Table

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO 30:
Drop staging
DROP TABLE IF EXISTS workspace.wc_badge.c_wc_badge_d_stg;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO 40: Create staging
CREATE TABLE workspace.wc_badge.c_wc_badge_d_stg
(
    INTEGRATION_ID      STRING,
    BADGE_ID            STRING,
    BADGE_STATUS        DOUBLE,
    CONTACT_EMAIL       STRING,
    ORG_NAME            STRING,
    CREATED_DATE        TIMESTAMP,
    LAST_UPDATED_DATE   TIMESTAMP,
    ETL_PROC_WID        DOUBLE
)
USING DELTA;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO 50: Load staging with incremental logic
INSERT INTO workspace.wc_badge.c_wc_badge_d_stg
SELECT 
    b.INTEGRATION_ID,
    b.BADGE_ID,
    b.STATUS,
    b.CONTACT_EMAIL,
    b.ORGANISATION_NAME,
    b.CREATION_DATE,
    b.LAST_UPDATE_DATE,
    CAST((SELECT etl_proc_wid FROM v_etl_parameters) AS DOUBLE) AS ETL_PROC_WID
FROM workspace.wc_badge.src_badge_table AS b
WHERE b.LAST_UPDATE_DATE > to_date('2024-01-01', 'yyyy-MM-dd')
  AND b.LAST_UPDATE_DATE <= current_date();

In [ ]:
-- MAGIC %sql
SELECT COUNT(*) AS record_count
FROM workspace.wc_badge.c_wc_badge_d_stg;

## Merge into Target

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO 100: Merge into target
MERGE INTO workspace.wc_badge.wc_badge_details_d AS T
USING (
    SELECT 
        INTEGRATION_ID,
        BADGE_ID,
        BADGE_STATUS        AS STATUS,
        CONTACT_EMAIL,
        ORG_NAME,
        CREATED_DATE,
        LAST_UPDATED_DATE
    FROM workspace.wc_badge.c_wc_badge_d_stg
) AS S
ON (T.INTEGRATION_ID = S.INTEGRATION_ID)
WHEN MATCHED THEN
    UPDATE SET
        T.STATUS          = S.STATUS,
        T.CONTACT_EMAIL   = S.CONTACT_EMAIL,
        T.ORG_NAME        = S.ORG_NAME,
        T.W_UPDATE_DT     = current_timestamp()
WHEN NOT MATCHED THEN
    INSERT (
        BADGE_ID,
        STATUS,
        CONTACT_EMAIL,
        ORG_NAME,
        INTEGRATION_ID,
        W_INSERT_DT,
        W_UPDATE_DT
    ) VALUES (
        S.BADGE_ID,
        S.STATUS,
        S.CONTACT_EMAIL,
        S.ORG_NAME,
        S.INTEGRATION_ID,
        current_timestamp(),
        current_timestamp()
    );

## Cleanup

In [ ]:
-- MAGIC %sql
DROP TABLE IF EXISTS workspace.wc_badge.c_wc_badge_d_stg;

## Validation

In [ ]:
-- MAGIC %sql
SELECT COUNT(*) AS final_target_record_count
FROM workspace.wc_badge.wc_badge_details_d;

## Conversion Notes and Manual Actions Required

1.  **Schema Inference:** Inferred `wc_badge` as the target schema and `wc_badge` for the source, mapping to `workspace.wc_badge`.
2.  **ETL_PROC_WID:** The `ETL_PROC_WID` was hardcoded to `12345` in the source `INSERT` statement. It has been converted to a Databricks widget for parameterization. The column type `NUMBER` in the `CREATE TABLE` for staging has been mapped to `DOUBLE`, so the widget value is `CAST(${ETL_PROC_WID} AS DOUBLE)`.
3.  **Oracle `NUMBER` Type:** `NUMBER` without specified precision or scale has been mapped to `DOUBLE`. If higher precision or integer-like behavior is strictly required (e.g., `NUMBER(20,0)`), `DECIMAL(38,10)` or `BIGINT` should be used respectively.
4.  **Date Literals and Functions:** `TO_DATE('2024-01-01', 'YYYY-MM-DD')` and `SYSDATE` have been converted to `to_date('2024-01-01', 'yyyy-MM-dd')` and `current_date()` respectively, reflecting Spark SQL syntax and format string conventions.
5.  **Target Table DDL:** The DDL for `WC_BADGE_DETAILS_D` was not provided in the source. Ensure `workspace.wc_badge.wc_badge_details_d` exists and has the necessary columns with compatible data types (e.g., `STATUS DOUBLE`, `CONTACT_EMAIL STRING`, `ORG_NAME STRING`, `W_UPDATE_DT TIMESTAMP`, `BADGE_ID STRING`, `INTEGRATION_ID STRING`, `W_INSERT_DT TIMESTAMP`). If not, a `CREATE TABLE IF NOT EXISTS` statement for the target should be added before the MERGE, or the table should be created manually.